# Encoder Fine-tuning — bge-small-en-v1.5 (Exp_2)

Fine-tunea el encoder con **TripletLoss** sobre 622 hard-negative triplets.
Guarda el encoder fine-tuned en Drive para que `rag.ipynb` lo use.

**Corre en Colab T4 — ~15-20 min.**

Archivos a subir a Drive antes de correr:
- `Data/consultas_centro_control.json`
- `Data/knowledge_base/` (carpeta completa)

Output: `encoder_finetuned/` guardado en Drive.

In [ ]:
# ── CELL 0: Config ──────────────────────────────────────────────────────────
DRIVE_PATH   = "/content/drive/MyDrive/MASTER/Tercer_semestre/NLP_2/Competencia/Exp_2"
ENCODER_BASE = "BAAI/bge-small-en-v1.5"
ENCODER_OUT  = "encoder_finetuned"   # directorio local en Colab → se sube a Drive al final

In [ ]:
# ── CELL 1: Mount Drive + Install ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run([
    "pip", "install", "-q",
    "sentence-transformers>=3.0.0",
    "datasets>=2.20.0",
], check=True)

print("\n=== IMPORTANTE ===")
print("Reinicia el runtime (Runtime → Restart runtime) y continúa desde la siguiente celda.")

In [ ]:
# ── CELL 2: Imports (después del restart) ───────────────────────────────────
import os, json, shutil
import torch
from pathlib import Path
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import TripletLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# ── CELL 3: Copiar archivos necesarios desde Drive ───────────────────────────
os.makedirs("Data", exist_ok=True)

# consultas con hard negatives
src = os.path.join(DRIVE_PATH, "Data/consultas_centro_control.json")
if not os.path.exists("Data/consultas_centro_control.json"):
    shutil.copy(src, "Data/consultas_centro_control.json")
    print("Copiado: consultas_centro_control.json")

# knowledge base (textos completos de docs para triplets)
kb_dst = "Data/knowledge_base"
if not os.path.exists(kb_dst):
    shutil.copytree(os.path.join(DRIVE_PATH, "Data/knowledge_base"), kb_dst)
    print("Copiado: Data/knowledge_base/")

print("Archivos listos.")

In [ ]:
# ── CELL 4: Cargar texto completo de cada doc KB ────────────────────────────
docs = {}
kb_root = Path("Data/knowledge_base/knowledge_base")
for doc_path in sorted(kb_root.rglob("doc.md")):
    docs[doc_path.parent.name] = doc_path.read_text(encoding="utf-8")

print(f"Documentos KB: {len(docs)}")

In [ ]:
# ── CELL 5: Construir triplets (query, doc_correcto, doc_incorrecto) ─────────
with open("Data/consultas_centro_control.json") as f:
    consultas = json.load(f)

triplet_data = {"anchor": [], "positive": [], "negative": []}
skipped = 0

for item in consultas:
    if "hard_negative_doc_id" not in item:
        continue
    pos = docs.get(item["doc_id"], "")
    neg = docs.get(item["hard_negative_doc_id"], "")
    if not pos or not neg:
        skipped += 1
        continue
    triplet_data["anchor"].append(item["query"])
    triplet_data["positive"].append(pos)
    triplet_data["negative"].append(neg)

train_dataset = Dataset.from_dict(triplet_data)
print(f"Triplets: {len(train_dataset)} (saltados: {skipped})")

In [ ]:
# ── CELL 6: Fine-tuning con TripletLoss ─────────────────────────────────────
model_ft = SentenceTransformer(ENCODER_BASE)
loss     = TripletLoss(model=model_ft)

train_args = SentenceTransformerTrainingArguments(
    output_dir=ENCODER_OUT,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.10,
    fp16=True,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=10,
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model_ft,
    args=train_args,
    train_dataset=train_dataset,
    loss=loss,
)

trainer.train()
model_ft.save(ENCODER_OUT)
print(f"Encoder guardado en: {ENCODER_OUT}/")

In [ ]:
# ── CELL 7: Subir encoder fine-tuned a Drive ─────────────────────────────────
encoder_dst = os.path.join(DRIVE_PATH, ENCODER_OUT)
if os.path.exists(encoder_dst):
    shutil.rmtree(encoder_dst)
shutil.copytree(ENCODER_OUT, encoder_dst)
print(f"Guardado en Drive: {encoder_dst}/")
print("\nListo. Ahora corre rag.ipynb en Colab para reconstruir el índice con el encoder fine-tuned.")